# Pump It Up

## High Level Description

This code was developed as part of the [Pump It Up Competition](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/page/23/), on DrivenData. The goal of this ML project is to promote access to clean and portable water accross Tanzania. 

The model being developed is an ensemble model, utilizing XGBoost, and CatBoost.

The final ML model will predict whether water pumps should be labelled as "functional", "functional needs repair", or "non functional".  

## Table of Contents
1. [Import and Setup](#import-and-setup)
2. [Data Cleaning and Feature Engineering](#data-cleaning--feature-engineering)
3. [Convert Data Types](#convert-data-types)
4. [Training and Hpyerparameter Tuning](#training-and-hyperparameter-tuning)
   1. [Random Forest](#random-forest)
   2. [XGBoost](#xgboost)
   3. [CatBoost](#catboost)
5. [Ensemble Voting](#ensemble-voting)

### Import and Setup
Import the standard python libraries and datasets.

To download a copy of the dataset for yourself, please sign up for the [Pump It Up Competition](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/page/23/) on DrivenData, and download it for free.

In [ ]:
import numpy as np
import pandas as pd
import HelperFunctions

In [ ]:
xtrain = pd.read_csv(r'D:\MMAI\Pump-It-Up\data\x_train.csv')
xtest = pd.read_csv(r'D:\MMAI\Pump-It-Up\data\x_test.csv')
ytrain = pd.read_csv(r'D:\MMAI\Pump-It-Up\data\y_train.csv')
raw = pd.concat([xtrain, xtest])

In [ ]:
del ytrain['id']

### Data Cleaning & Feature Engineering

In [ ]:
# cleanup - make sure everything is lowercase
raw = HelperFunctions.toLowerCase(raw)

In [ ]:
# FE - Handle DateType variables
# Convert date into year, month, and days since recorded

def fixDates(df):
    df['date_recorded'] = pd.to_datetime(df['date_recorded'])
    df['year_recorded'] = (df['date_recorded'].dt.year).astype('int')
    df['month_recorded'] = (df['date_recorded'].dt.month).astype('str')
    df['days_since_recorded'] = ((pd.to_datetime('2001-03-26') - df['date_recorded']).dt.days).astype('int')
    del df['date_recorded']
    return df

xtrain = fixDates(xtrain)
xtest = fixDates(xtest)
raw = fixDates(raw)

In [ ]:
# Impute 'longitude' based on subvillage, ward, lga, region
raw = HelperFunctions.imputeLong(raw)

# Impute 'permit' based on subvillage, ward, lga, region
permit_geo_mode = raw.groupby(['subvillage', 'ward', 'lga', 'region'])['permit'].agg(pd.Series.mode).reset_index()
permit_geo_mode = permit_geo_mode.rename(columns={"permit": "imputed_permit_geo"})
raw = raw.merge(permit_geo_mode, how='left', on=['subvillage', 'ward', 'lga', 'region'])
raw['imputed_permit'] = np.where(raw['permit'].isna(), raw['imputed_permit_geo'], raw['permit'])
raw['imputed_permit'] = np.where(raw['imputed_permit'].isna(), raw['permit'].mode()[0], raw['imputed_permit'])
raw = raw.drop(['permit', 'imputed_permit_geo'], axis=1)

# Impute 'population' based on subvillage, ward, lga, region
population_geo_mode = raw.groupby(['subvillage', 'ward', 'lga', 'region'])['population'].agg(pd.Series.mode).reset_index()
population_geo_mode = population_geo_mode.rename(columns={"population": "imputed_population_geo"})
raw = raw.merge(population_geo_mode, how='left', on=['subvillage', 'ward', 'lga', 'region'])
raw['imputed_population'] = np.where(raw['population'].isna(), raw['imputed_population_geo'], raw['population'])
raw['imputed_population'] = np.where(raw['imputed_population'].isna(), raw['population'].mode()[0], raw['imputed_population'])
raw = raw.drop(['population', 'imputed_population_geo'], axis=1)

# Impute 'gps_height' based on basin, subvillage, ward, lga, region
gps_height_geo_mode = raw.groupby(['basin', 'subvillage', 'ward', 'lga', 'region'])['gps_height'].agg(pd.Series.mode).reset_index()
gps_height_geo_mode = gps_height_geo_mode.rename(columns={"gps_height": "imputed_gps_height_geo"})
raw = raw.merge(gps_height_geo_mode, how='left', on=['basin','subvillage', 'ward', 'lga', 'region'])
raw['imputed_gps_height'] = np.where(raw['gps_height'].isna(), raw['imputed_gps_height_geo'], raw['gps_height'])
raw['imputed_gps_height'] = np.where(raw['imputed_gps_height'].isna(), raw['gps_height'].mode()[0], raw['imputed_gps_height'])
raw = raw.drop(['gps_height', 'imputed_gps_height_geo'], axis=1)

# Impute public meeting based on mode
raw['public_meeting']=raw['public_meeting'].fillna(raw['public_meeting'].mode()[0])

# Impute construction_year by mode
raw['construction_year'] = raw['construction_year'].fillna(raw['construction_year'].mode()[0])




In [ ]:
# FE - Binning
# Group columns with high feature cardinality

cols = [i for i in raw.columns if type(raw[i].iloc[0]) == str]
raw[cols] = raw[cols].where(raw[cols].apply(lambda x: x.map(x.value_counts())) > 100, "other")
for column in cols:
    for i in raw[column].unique():
        if i not in raw[column].unique():
            raw[column].replace(i, 'other', inplace=True)

In [ ]:
# Cleanup - Drop columns that are poorly distributed or have high co-linearity

# Poorly distributed
xtrain, xtest = HelperFunctions.removeCol(xtrain, xtest, 'amount_tsh')
xtrain, xtest = HelperFunctions.removeCol(xtrain, xtest, 'wpt_name')
xtrain, xtest = HelperFunctions.removeCol(xtrain, xtest, 'num_private')

# High co-linearity
xtrain, xtest = HelperFunctions.removeCol(xtrain, xtest, 'quantity_group')
xtrain, xtest = HelperFunctions.removeCol(xtrain, xtest, 'region')
xtrain, xtest = HelperFunctions.removeCol(xtrain, xtest, 'waterpoint_type_group')
xtrain, xtest = HelperFunctions.removeCol(xtrain, xtest, 'extraction_type_group')
xtrain, xtest = HelperFunctions.removeCol(xtrain, xtest, 'source_type')


### Convert Data Types

In [ ]:

#replace string to integer
raw['public_meeting'] = raw['public_meeting'].replace({True: 1, False: 0})
raw['imputed_permit'] = raw['imputed_permit'].apply(lambda x: 0 if isinstance(x, list) and not x else x)
raw['imputed_permit'] = raw['imputed_permit'].apply(lambda x: x if x in [True, False] else 0)
raw['imputed_permit'] = raw['imputed_permit'].replace({True: 1, False: 0})


#change to integer
raw[['imputed_gps_height', 'construction_year', 'imputed_population']] = raw[['imputed_gps_height', 'construction_year', 'imputed_population']].astype('int')

#change type to categorical
raw[[ 'region_code', 'district_code', 'num_private']] = raw[[ 'region_code', 'district_code', 'num_private']].astype('str')

#remove decimal
raw['district_code'] = raw['district_code'].str.split(".").str[0]

raw= raw.rename(columns={"imputed_permit": "permit",
                    "imputed_gps_height": "gps_height", 
                   'imputed_population': 'population', 'imputed_longitude': 'longitude'}, errors="raise")


In [ ]:
del raw['amount_tsh']
del raw['wpt_name']
del raw['num_private']
del raw['quantity_group']
del raw['region']
del raw['waterpoint_type_group']
del raw['extraction_type_group']
del raw['source_type']


In [ ]:
raw.info()

In [ ]:
x_train = raw[raw['id'].isin(xtrain['id'])]
x_test = raw[raw['id'].isin(xtest['id'])]
del x_train['id']
del x_test['id']

In [ ]:
def oneHotEncoding(X_train, X_test):
    columns = [i for i in X_train.columns if type(X_train[i].iloc[0]) == str]
    for column in columns:
        X_train[column].fillna('NULL', inplace = True)
        good_cols = [column+'_'+i for i in X_train[column].unique() if i in X_test[column].unique()]
        X_train = pd.concat((X_train, pd.get_dummies(X_train[column], prefix = column)[good_cols]), axis = 1)
        X_test = pd.concat((X_test, pd.get_dummies(X_test[column], prefix = column)[good_cols]), axis = 1)
        del X_train[column]
        del X_test[column]
    return X_train, X_test

x_train, x_test = oneHotEncoding(x_train, x_test)

### Training and Hyperparameter Tuning

Trained and tested the following Models: Random Forest, XGBoost, LGBM, CatBoost. Hypertuned with Optuna.

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_validate, y_train, y_validate = train_test_split(x_train, ytrain, test_size=0.2, random_state=42)

#### Random Forest

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(criterion='gini',
                                max_features='sqrt',
                                min_samples_split=6,
                                oob_score=True,
                                random_state=1,
                                n_jobs=-1)

# param_grid = {"n_estimators" : [500, 750, 1000]}
# param_grid = {"n_estimators" : [1000]}

def objective(trial):
    criterion = trial.suggest_categorical('criterion', ['gini'])
    n_estimators = 700
    # n_estimators = trial.suggest_int('n_estimators', 500, 1500)
    max_depth = trial.suggest_int('max_depth', 10, 100, 10)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 40)
    max_features = trial.suggest_categorical('max_features', ['sqrt'])

    clf = RandomForestClassifier(
        n_estimators=n_estimators, 
        max_depth=max_depth, 
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=42
    )
    
    return cross_val_score(clf, x_train, ytrain.values.ravel(), cv=5, scoring='accuracy', verbose=1, error_score='raise', n_jobs=-2).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5, n_jobs=-1)

print('Best trial: score {}, params {}'.format(study.best_trial.value, study.best_trial.params))

In [ ]:
from sklearn.metrics import accuracy_score
top_params = study.best_params

# Previously found best parameters
# top_params = {'criterion': 'gini', 'n_estimators': 958, 'max_depth': 30, 'min_samples_split': 5, 'max_features': 'sqrt'}
# [I 2023-12-18 11:03:19,199] Trial 15 finished with value: 0.8129966329966329 and parameters: {'criterion': 'gini', 'n_estimators': 958, 'max_depth': 30, 'min_samples_split': 5, 'max_features': 'sqrt'}. Best is trial 15 with value: 0.8129966329966329.

# Train a model on the entire training dataset using the top parameters
final_model = RandomForestClassifier(**top_params, random_state=42, n_jobs=-1)
final_model.fit(x_train, ytrain.values.ravel())

# Make predictions on the test set
test_predictions = final_model.predict(x_validate)

# Optionally, evaluate the predictions
test_accuracy = accuracy_score(y_validate, test_predictions)
print(f'Test Accuracy: {test_accuracy}')

In [ ]:
# predictions = best_clf.predict(X_test)
import os
y_test = pd.read_csv('data\y_test.csv')
pred = pd.DataFrame(test_predictions, columns = [y_test.columns[1]])
del y_test['status_group']
y_test = pd.concat((y_test, pred), axis = 1)
y_test.to_csv(os.path.join('data', 'y_test101.csv'), sep=",", index = False)

#### XGBoost

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
ytrain_encoded = le.fit_transform(y_train.values.ravel())

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 750, 1250),
        'max_depth': trial.suggest_int('max_depth', 1, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 70),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10, step=0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10, step=0.5),
        'subsample': trial.suggest_float('subsample', 0.5, 1, step=0.05),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1, step=0.05),
        'gamma': trial.suggest_int('gamma', 0, 5),
        'max_bin': trial.suggest_int('max_bin', 50, 100),
        'objective': 'multi:softmax'
    }
    
    clf = XGBClassifier(**params)
    
    return cross_val_score(clf, x_train, ytrain_encoded, cv=2, verbose=1, scoring='accuracy').mean()

study2 = optuna.create_study(direction='maximize')
study2.optimize(objective, n_trials=50, n_jobs=-2)


In [ ]:
print('Best trial: score {}, params {}'.format(study2.best_trial.value, study2.best_trial.params))

from sklearn.metrics import accuracy_score
top_params2 = study2.best_params

# Previously found best parameters
# [I 2023-12-18 11:03:19,199] Trial 15 finished with value: 0.8129966329966329 and parameters: {'criterion': 'gini', 'n_estimators': 958, 'max_depth': 30, 'min_samples_split': 5, 'max_features': 'sqrt'}. Best is trial 15 with value: 0.8129966329966329.
# [Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.

# Train a model on the entire training dataset using the top parameters
final_model2 = XGBClassifier(**top_params2, random_state=42, n_jobs=-1)
final_model2.fit(x_train, ytrain_encoded)

# Make predictions on the test set
test_predictions2 = final_model2.predict(x_validate)

# Optionally, evaluate the predictions
test_accuracy2 = accuracy_score(y_validate, test_predictions)
print(f'Test Accuracy: {test_accuracy2}')

In [ ]:
# predictions = best_clf.predict(X_test)
import os
y_test = pd.read_csv('data\y_test.csv')
decoded_predictions = le.inverse_transform(test_predictions2)
pred = pd.DataFrame(decoded_predictions, columns = [y_test.columns[1]])
del y_test['status_group']
y_test = pd.concat((y_test, pred), axis = 1)
y_test.to_csv(os.path.join('data', 'y_test102.csv'), sep=",", index = False)

#### CatBoost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score
import optuna

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 750, 1250),
        # 'n_estimators': trial.suggest_int('n_estimators', 750, 1250),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_strength': trial.suggest_int('random_strength', 0, 100),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'od_type': trial.suggest_categorical('od_type', ['IncToDec', 'Iter']),
        'od_wait': trial.suggest_int('od_wait', 10, 50),
        'max_bin': trial.suggest_int('max_bin', 50, 100)
    }
    
    clf = CatBoostClassifier(**params, verbose=1)
    
    return cross_val_score(clf, x_train, y_train.values.ravel(), cv=2, scoring='accuracy').mean()

study3 = optuna.create_study(direction='maximize')
study3.optimize(objective, n_trials=50, n_jobs=10)

print('Best trial: score {}, params {}'.format(study3.best_trial.value, study3.best_trial.params))

In [ ]:
print('Best trial: score {}, params {}'.format(study3.best_trial.value, study3.best_trial.params))

from sklearn.metrics import accuracy_score
top_params3 = study3.best_params

# Previously found best parameters
# [I 2023-12-18 11:03:19,199] Trial 15 finished with value: 0.8129966329966329 and parameters: {'criterion': 'gini', 'n_estimators': 958, 'max_depth': 30, 'min_samples_split': 5, 'max_features': 'sqrt'}. Best is trial 15 with value: 0.8129966329966329.
# [Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.

# Train a model on the entire training dataset using the top parameters
final_model3 = CatBoostClassifier(**top_params3, random_state=42)
final_model3.fit(x_train, y_train)

# Make predictions on the test set
test_predictions3 = final_model3.predict(x_validate)

# Optionally, evaluate the predictions
test_accuracy3 = accuracy_score(y_validate, test_predictions3)
print(f'Test Accuracy: {test_accuracy3}')

In [ ]:
import os
y_test = pd.read_csv(r'D:\MMAI\Pump-It-Up\data\y_test.csv')
pred = pd.DataFrame(test_predictions3, columns = [y_test.columns[1]])
del y_test['status_group']
y_test = pd.concat((y_test, pred), axis = 1)
y_test.to_csv(os.path.join('data', 'y_test102.csv'), sep=",", index = False)

### Ensemble Voting

In [ ]:
import pandas as pd
from scipy.stats import mode

# Load the prediction files
y_test_RF = pd.read_csv('data/y_test_RF.csv')
y_test_XG = pd.read_csv('data/y_test_XG.csv')
y_test_Cat = pd.read_csv('data/y_test_Cat.csv')

# Extract the predictions
pred_RF = y_test_RF['status_group']
pred_XG = y_test_XG['status_group']
pred_Cat = y_test_Cat['status_group']

# Perform majority voting
majority_vote = mode([pred_RF, pred_XG, pred_Cat])[0][0]

# Create a new DataFrame for the majority vote predictions
majority_vote_df = pd.DataFrame(majority_vote, columns=['status_group'])

# Add the 'id' field from the original dataframes
majority_vote_df['id'] = pred_RF['id']

# Rearrange the columns to make 'id' the first column
majority_vote_df = majority_vote_df.reindex(columns=['id', 'status_group'])

# Save the majority vote predictions to a CSV file
majority_vote_df.to_csv('data/y_test_majority_vote.csv', index=False)